# Pipeline de notas: Excel → SQL Server

Este cuaderno toma el reporte **"Estudiantes notas y observaciones"** (.xlsx), lo limpia y lo carga en la tabla `dbo.notas_observaciones` (una fila por curso / estudiante / corte / componente).

| Paso | Qué hace |
|---|---|
| 0 | Instalar librerías (solo la primera vez) |
| 1 | Configuración: carpeta de los Excel y datos del servidor |
| 2 | Importar librerías y buscar los archivos |
| 3 | Leer el Excel crudo y mirarlo |
| 4 | Renombrar columnas al nombre que tienen en la tabla |
| 5 | Limpiar textos (espacios, tildes) |
| 6 | Limpiar documentos: `C.C.` → `CC` |
| 7 | Sacar el código del curso (para cruzar con asistencia) |
| 8 | Separar el subparámetro en **corte** y **componente** |
| 9 | Convertir números: porcentajes, notas, fechas, sí/no |
| 10 | Armar la tabla final y quitar duplicados entre archivos |
| 11 | Validar antes de cargar |
| 12 | Conectarse a SQL Server |
| 13 | Cargar con `MERGE` |
| 14 | Verificar en la base de datos (y cruzar con asistencia) |

**Diferencia con el cuaderno de asistencia:** aquí el Excel **ya viene en formato largo** (una fila por subparámetro), así que no hay que "girarlo". El trabajo es casi todo **limpiar y convertir columnas**.

> Ejecuta las celdas en orden con **Shift + Enter**, o todo con **Run All**.

CARPETA  = r"C:\Users\sebastian.mendez\OneDrive - CENTRO COLOMBO AMERICANO\Escritorio\Automatizacion Oxford\Descargas\Q10\notas y observaciones"   # <- cambia esto
PATRON   = "*notas*.xlsx"

SERVIDOR = "bi-db-group.database.windows.net"
BASE     = "rendimiento_academico"
USUARIO  = "bi_admin"
CLAVE    = "53/O~D&2m#o}"
DRIVER   = "ODBC Driver 18 for SQL Server"

CARGAR_EN_BD = True                         # True cuando ya hayas revisado todo

In [30]:
# %pip install pandas openpyxl sqlalchemy pyodbc

## Paso 1 · Configuración

**Única celda que tienes que modificar.**

- `CARPETA`: dónde guardas los Excel de notas.
- `PATRON`: qué archivos tomar de esa carpeta. `*notas*.xlsx` = cualquier .xlsx cuyo nombre contenga "notas".
- `SERVIDOR` y `BASE`: ya quedaron con los tuyos.
- `CARGAR_EN_BD`: `False` para probar sin tocar la base; `True` para cargar.

In [31]:
CARPETA  = r"C:\Users\sebastian.mendez\OneDrive - CENTRO COLOMBO AMERICANO\Escritorio\Automatizacion Oxford\Descargas\Q10\notas y observaciones"   # <- cambia esto
PATRON   = "*notas*.xlsx"

SERVIDOR = "bi-db-group.database.windows.net"
BASE     = "rendimiento_academico"
USUARIO  = "bi_admin"
CONTRASENA_BD = "53/O~D&2m#o}"
DRIVER   = "ODBC Driver 18 for SQL Server"

CARGAR_EN_BD = True                         # True cuando ya hayas revisado todo

## Paso 2 · Importar librerías y buscar archivos

In [32]:
import re
import unicodedata
from datetime import datetime
from decimal import Decimal
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 40)

archivos = sorted(Path(CARPETA).glob(PATRON), key=lambda p: p.stat().st_mtime)  # del más viejo al más nuevo
print(f"Encontré {len(archivos)} archivo(s):")
for a in archivos:
    print("  -", a.name, "| modificado:", datetime.fromtimestamp(a.stat().st_mtime).strftime("%Y-%m-%d %H:%M"))

Encontré 1 archivo(s):
  - Estudiantes_notas_y_observaciones (5).xlsx | modificado: 2026-10-16 10:43


## Paso 3 · Leer el Excel crudo

Leemos **todos** los archivos encontrados y los juntamos en una sola tabla. A cada fila le anotamos de qué archivo vino (`archivo_origen`).

**Detalle importante:** el número de documento se lee como **texto** (`dtype=str`). Si pandas lo leyera como número, perderíamos los ceros a la izquierda y documentos con letras darían error.

In [33]:
partes = []
for ruta in archivos:
    parte = pd.read_excel(ruta, dtype={"Número identificación": str})
    parte["archivo_origen"] = ruta.name
    partes.append(parte)
    print(f"{ruta.name}: {len(parte):,} filas")

crudo = pd.concat(partes, ignore_index=True)
print(f"\nTotal: {len(crudo):,} filas × {crudo.shape[1]} columnas")
crudo.head(3)

Estudiantes_notas_y_observaciones (5).xlsx: 64,880 filas

Total: 64,880 filas × 24 columnas


,Estudiante,Tipo identificación,Número identificación,Sede - Jornada,Programa,Asignatura,Período,Código curso,Curso,Docente,Fecha inicio curso,Fecha fin curso,Archivado,Parámetro,Porcentaje parámetro,Nota parámetro,Subparámetro,Porcentaje subparámetro,Nota subparámetro,Promedio,Acumulado,Inasistencias,Observaciones,archivo_origen
0,Abdor Pacheco Yessenia,C.C.,1014979993,B2G - Única,YES BOGOTA 2026-2,SDDE26-2_B1-1,B2G_YES BOGOTA-2 C1_SEP_2026,-6AM7 SDDE_2026-2,B1.1-6AM7 SDDE_2026-2,Cortes Bello Liz Alejandra,2026-09-07,2026-11-13,No,Corte 1,50%,3.7,Communication 1,20%,3.5,No aprob.,50%,12,Primer Corte: Has demostrado una com...,Estudiantes_notas_y_observaciones (5...
1,Abdor Pacheco Yessenia,C.C.,1014979993,B2G - Única,YES BOGOTA 2026-2,SDDE26-2_B1-1,B2G_YES BOGOTA-2 C1_SEP_2026,-6AM7 SDDE_2026-2,B1.1-6AM7 SDDE_2026-2,Cortes Bello Liz Alejandra,2026-09-07,2026-11-13,No,Corte 1,50%,3.7,Language 1,20%,3.5,No aprob.,50%,12,Primer Corte: Has demostrado una com...,Estudiantes_notas_y_observaciones (5...
2,Abdor Pacheco Yessenia,C.C.,1014979993,B2G - Única,YES BOGOTA 2026-2,SDDE26-2_B1-1,B2G_YES BOGOTA-2 C1_SEP_2026,-6AM7 SDDE_2026-2,B1.1-6AM7 SDDE_2026-2,Cortes Bello Liz Alejandra,2026-09-07,2026-11-13,No,Corte 1,50%,3.7,Learning 1,20%,3.5,No aprob.,50%,12,Primer Corte: Has demostrado una com...,Estudiantes_notas_y_observaciones (5...


## Paso 4 · Renombrar columnas

Pasamos de los nombres del Excel (con tildes y espacios) a los nombres de la tabla en SQL Server.

Si algún día el Excel trae una columna con otro nombre, esta celda **se detiene y te dice cuál falta**, en lugar de cargar datos incompletos.

`Código curso` (`-7PM215 SDDE_2026-2`) no se usa: es una versión recortada de `Curso` y no aporta nada nuevo.

In [34]:
COLUMNAS_EXCEL = {
    "Estudiante":              "nombre_completo",
    "Tipo identificación":     "tipo_documento",
    "Número identificación":   "numero_documento",
    "Sede - Jornada":          "sede_jornada",
    "Programa":                "programa",
    "Asignatura":              "asignatura",
    "Período":                 "periodo",
    "Curso":                   "curso_nombre",
    "Docente":                 "docente",
    "Fecha inicio curso":      "fecha_inicio",
    "Fecha fin curso":         "fecha_fin",
    "Archivado":               "archivado",
    "Parámetro":               "parametro",
    "Porcentaje parámetro":    "porcentaje_corte",
    "Nota parámetro":          "nota_corte",
    "Subparámetro":            "subparametro",
    "Porcentaje subparámetro": "porcentaje_componente",
    "Nota subparámetro":       "nota",
    "Promedio":                "promedio",
    "Acumulado":               "acumulado_pct",
    "Inasistencias":           "inasistencias",
    "Observaciones":           "observaciones",
}

faltan = [c for c in COLUMNAS_EXCEL if c not in crudo.columns]
if faltan:
    raise ValueError(f"Al Excel le faltan estas columnas: {faltan}")

df = crudo.rename(columns=COLUMNAS_EXCEL)[list(COLUMNAS_EXCEL.values()) + ["archivo_origen"]].copy()
df.columns.tolist()

['nombre_completo',
 'tipo_documento',
 'numero_documento',
 'sede_jornada',
 'programa',
 'asignatura',
 'periodo',
 'curso_nombre',
 'docente',
 'fecha_inicio',
 'fecha_fin',
 'archivado',
 'parametro',
 'porcentaje_corte',
 'nota_corte',
 'subparametro',
 'porcentaje_componente',
 'nota',
 'promedio',
 'acumulado_pct',
 'inasistencias',
 'observaciones',
 'archivo_origen']

## Paso 5 · Limpiar textos

Las mismas funciones del cuaderno de asistencia: quitan espacios dobles, espacios "raros" (`\xa0`) y espacios al inicio o al final. Si una celda queda vacía, se vuelve `None` (que en SQL Server será `NULL`).

Ejemplo real de tu archivo: algunas observaciones terminan con dos espacios (`"...el docente.  "`).

In [35]:
def limpiar_texto(valor):
    if valor is None or (isinstance(valor, float) and pd.isna(valor)) or valor is pd.NA:
        return None
    texto = unicodedata.normalize("NFC", str(valor)).replace("\xa0", " ")
    texto = re.sub(r"[ \t]+", " ", texto).strip()   # en observaciones respetamos los saltos de línea
    return texto or None

COLUMNAS_TEXTO = ["nombre_completo", "tipo_documento", "numero_documento", "sede_jornada", "programa",
                  "asignatura", "periodo", "curso_nombre", "docente", "archivado", "parametro",
                  "porcentaje_corte", "subparametro", "porcentaje_componente", "promedio",
                  "acumulado_pct", "observaciones"]

for col in COLUMNAS_TEXTO:
    df[col] = df[col].map(limpiar_texto).astype(object)

print("Ejemplo:", repr(limpiar_texto("  Abril   Fonseca\xa0Manuel  ")))

Ejemplo: 'Abril Fonseca Manuel'


## Paso 6 · Limpiar documentos

El tipo viene con puntos (`C.C.`, `C.E.`, `T.I.`). Lo dejamos **igual que en `dbo.asistencia`** (`CC`, `CE`, `TI`); si no, el cruce entre las dos tablas no encontraría a los estudiantes.

Al número le quitamos puntos, espacios o guiones que se hayan colado.

In [36]:
TIPOS_DOC = {"CC": "CC", "TI": "TI", "CE": "CE", "PPT": "PPT", "PEP": "PEP",
             "PA": "PA", "PAS": "PA", "RC": "RC", "NIT": "NIT", "NUIP": "NUIP"}

print("Antes:", df["tipo_documento"].value_counts().to_dict())

tipo_sin_puntos = df["tipo_documento"].str.upper().str.replace(r"[^A-Z]", "", regex=True)
desconocidos = sorted(set(tipo_sin_puntos.dropna()) - set(TIPOS_DOC))
if desconocidos:
    raise ValueError(f"Tipos de documento que no conozco: {desconocidos}. Agrégalos a TIPOS_DOC.")

df["tipo_documento"]   = tipo_sin_puntos.map(TIPOS_DOC)
df["numero_documento"] = df["numero_documento"].str.replace(r"[^0-9A-Za-z]", "", regex=True).str.upper()

print("Después:", df["tipo_documento"].value_counts().to_dict())

Antes: {'C.C.': 64560, 'PPT': 230, 'C.E.': 60, 'NUIP': 20, 'T.I.': 10}
Después: {'CC': 64560, 'PPT': 230, 'CE': 60, 'NUIP': 20, 'TI': 10}


## Paso 7 · Código del curso

En notas el curso viene como `B2.1-7PM215 SDDE_2026-2`, pero en `dbo.asistencia` es `B2.1-7PM215`. Tomamos **la parte antes del primer espacio** para que las dos tablas usen el mismo código y se puedan cruzar.

In [37]:
df["curso_codigo"] = df["curso_nombre"].str.split(" ").str[0]

df[["curso_nombre", "curso_codigo"]].drop_duplicates().head()

,curso_nombre,curso_codigo
0,B1.1-6AM7 SDDE_2026-2,B1.1-6AM7
10,B1.2-8AM85 SDDE_2026-2,B1.2-8AM85
20,B2.1-7PM215 SDDE_2026-2,B2.1-7PM215
30,B1.2-5PM108 SDDE_2026-2,B1.2-5PM108
40,B2.1-7PM206 SDDE_2026-2,B2.1-7PM206


## Paso 8 · Separar el subparámetro en corte y componente

`Subparámetro = "Communication 1"` en realidad son **dos datos**: el componente (`Communication`) y el corte (`1`).

- **corte**: el número que viene en `Parámetro` (`"Corte 1"` → `1`).
- **componente**: el subparámetro **sin el número final** (`"Communication 1"` → `"Communication"`).

Además, tu archivo trae `Task accomplishment 1` pero `Task Accomplishment 2` (con A mayúscula). Para que sean **el mismo componente**, agrupamos sin importar mayúsculas y nos quedamos con la escritura más frecuente.

Guardamos también el texto original en `subparametro`, por si alguna vez necesitas verlo tal cual.

In [38]:
# corte: el número dentro de "Corte 1"
df["corte"] = pd.to_numeric(df["parametro"].str.extract(r"(\d+)")[0], errors="coerce")
sin_numero = df.loc[df["corte"].isna(), "parametro"].unique()
if len(sin_numero):
    raise ValueError(f"Parámetros sin número de corte: {sin_numero}")
df["corte"] = df["corte"].astype(int)

# componente: quitamos el número final
componente_bruto = df["subparametro"].str.replace(r"\s*\d+$", "", regex=True)

# unificar mayúsculas: para cada versión en minúscula, la escritura más común
escritura = (componente_bruto.groupby(componente_bruto.str.lower())
             .agg(lambda s: s.value_counts().index[0]))
df["componente"] = componente_bruto.str.lower().map(escritura)

# revisión: el número del subparámetro debería coincidir con el corte
numero_sub = pd.to_numeric(df["subparametro"].str.extract(r"(\d+)$")[0], errors="coerce")
no_coincide = (numero_sub.notna() & (numero_sub != df["corte"])).sum()
print("Filas donde el número del subparámetro ≠ corte:", no_coincide)

df[["parametro", "subparametro", "corte", "componente"]].drop_duplicates().sort_values(["corte", "componente"])

Filas donde el número del subparámetro ≠ corte: 0


,parametro,subparametro,corte,componente
0,Corte 1,Communication 1,1,Communication
1,Corte 1,Language 1,1,Language
2,Corte 1,Learning 1,1,Learning
3,Corte 1,Task accomplishment 1,1,Task accomplishment
4,Corte 1,Workshops 1,1,Workshops
5,Corte 2,Communication 2,2,Communication
6,Corte 2,Language 2,2,Language
7,Corte 2,Learning 2,2,Learning
8,Corte 2,Task Accomplishment 2,2,Task accomplishment
9,Corte 2,Workshops 2,2,Workshops


## Paso 9 · Convertir números, fechas y sí/no

| Columna | Viene así | Queda así |
|---|---|---|
| Porcentajes | `'50%'` (texto) | `50.0` |
| Notas | `3.7` o vacío | `3.7` o `None` (**vacío ≠ 0**) |
| Inasistencias | `12` | `12` (entero) |
| Fechas | `2026-09-07 00:00:00` | `2026-09-07` (solo fecha) |
| Archivado | `'No'` / `'Sí'` | `False` / `True` |

In [39]:
def porcentaje(valor):
    # '50%' -> 50.0 ; también acepta 0.5 (por si Excel lo trae como número) -> 50.0
    if valor is None:
        return None
    texto = str(valor).replace("%", "").replace(",", ".").strip()
    numero = float(texto)
    return numero * 100 if "%" not in str(valor) and numero <= 1 else numero

for col in ["porcentaje_corte", "porcentaje_componente", "acumulado_pct"]:
    df[col] = df[col].map(porcentaje)

for col in ["nota", "nota_corte"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").round(2)

df["inasistencias"] = pd.to_numeric(df["inasistencias"], errors="coerce").astype("Int64")

for col in ["fecha_inicio", "fecha_fin"]:
    df[col] = pd.to_datetime(df[col], dayfirst=True, errors="coerce").dt.date

def a_si_no(valor):
    # 'Sí', 'SI', 'si' -> True ; 'No' o vacío -> False
    texto = "".join(c for c in unicodedata.normalize("NFD", str(valor or ""))
                    if unicodedata.category(c) != "Mn").strip().lower()
    return texto in ("si", "s", "true", "1")

df["archivado"] = df["archivado"].map(a_si_no)

print("Porcentajes de corte:      ", sorted(float(x) for x in df["porcentaje_corte"].dropna().unique()))
print("Porcentajes de componente: ", sorted(float(x) for x in df["porcentaje_componente"].dropna().unique()))
print(f"Notas calificadas: {df['nota'].notna().sum():,} | sin calificar: {df['nota'].isna().sum():,}")
print("Archivado:", df["archivado"].value_counts().to_dict())

Porcentajes de corte:       [50.0]
Porcentajes de componente:  [20.0]
Notas calificadas: 32,463 | sin calificar: 32,417
Archivado: {False: 64880}


## Paso 10 · Armar la tabla final

1. Dejamos las columnas **en el mismo orden que la tabla** `dbo.notas_observaciones`.
2. Si cargas **varios archivos** y un mismo estudiante aparece en más de uno (por ejemplo, el reporte de esta semana y el de la anterior), nos quedamos con la fila del **archivo más reciente**. Por eso en el paso 2 ordenamos los archivos del más viejo al más nuevo.

In [40]:
COLUMNAS_TABLA = ["curso_codigo", "curso_nombre", "asignatura", "docente", "programa", "periodo",
                  "sede_jornada", "fecha_inicio", "fecha_fin", "archivado",
                  "tipo_documento", "numero_documento", "nombre_completo",
                  "corte", "porcentaje_corte", "nota_corte", "subparametro", "componente",
                  "porcentaje_componente", "nota",
                  "promedio", "acumulado_pct", "inasistencias", "observaciones", "archivo_origen"]

CLAVE = ["curso_codigo", "tipo_documento", "numero_documento", "corte", "componente"]

final = df[COLUMNAS_TABLA].drop_duplicates(subset=CLAVE, keep="last").reset_index(drop=True)

print(f"Filas antes: {len(df):,} | después de quitar repetidos entre archivos: {len(final):,}")
print(f"Cursos: {final['curso_codigo'].nunique()} | Estudiantes: {final['numero_documento'].nunique():,}")
final.head()

Filas antes: 64,880 | después de quitar repetidos entre archivos: 64,880
Cursos: 217 | Estudiantes: 6,488


,curso_codigo,curso_nombre,asignatura,docente,programa,periodo,sede_jornada,fecha_inicio,fecha_fin,archivado,tipo_documento,numero_documento,nombre_completo,corte,porcentaje_corte,nota_corte,subparametro,componente,porcentaje_componente,nota,promedio,acumulado_pct,inasistencias,observaciones,archivo_origen
0,B1.1-6AM7,B1.1-6AM7 SDDE_2026-2,SDDE26-2_B1-1,Cortes Bello Liz Alejandra,YES BOGOTA 2026-2,B2G_YES BOGOTA-2 C1_SEP_2026,B2G - Única,2026-09-07,2026-11-13,False,CC,1014979993,Abdor Pacheco Yessenia,1,50.0,3.7,Communication 1,Communication,20.0,3.5,No aprob.,50.0,12,Primer Corte: Has demostrado una com...,Estudiantes_notas_y_observaciones (5...
1,B1.1-6AM7,B1.1-6AM7 SDDE_2026-2,SDDE26-2_B1-1,Cortes Bello Liz Alejandra,YES BOGOTA 2026-2,B2G_YES BOGOTA-2 C1_SEP_2026,B2G - Única,2026-09-07,2026-11-13,False,CC,1014979993,Abdor Pacheco Yessenia,1,50.0,3.7,Language 1,Language,20.0,3.5,No aprob.,50.0,12,Primer Corte: Has demostrado una com...,Estudiantes_notas_y_observaciones (5...
2,B1.1-6AM7,B1.1-6AM7 SDDE_2026-2,SDDE26-2_B1-1,Cortes Bello Liz Alejandra,YES BOGOTA 2026-2,B2G_YES BOGOTA-2 C1_SEP_2026,B2G - Única,2026-09-07,2026-11-13,False,CC,1014979993,Abdor Pacheco Yessenia,1,50.0,3.7,Learning 1,Learning,20.0,3.5,No aprob.,50.0,12,Primer Corte: Has demostrado una com...,Estudiantes_notas_y_observaciones (5...
3,B1.1-6AM7,B1.1-6AM7 SDDE_2026-2,SDDE26-2_B1-1,Cortes Bello Liz Alejandra,YES BOGOTA 2026-2,B2G_YES BOGOTA-2 C1_SEP_2026,B2G - Única,2026-09-07,2026-11-13,False,CC,1014979993,Abdor Pacheco Yessenia,1,50.0,3.7,Task accomplishment 1,Task accomplishment,20.0,3.5,No aprob.,50.0,12,Primer Corte: Has demostrado una com...,Estudiantes_notas_y_observaciones (5...
4,B1.1-6AM7,B1.1-6AM7 SDDE_2026-2,SDDE26-2_B1-1,Cortes Bello Liz Alejandra,YES BOGOTA 2026-2,B2G_YES BOGOTA-2 C1_SEP_2026,B2G - Única,2026-09-07,2026-11-13,False,CC,1014979993,Abdor Pacheco Yessenia,1,50.0,3.7,Workshops 1,Workshops,20.0,4.5,No aprob.,50.0,12,Primer Corte: Has demostrado una com...,Estudiantes_notas_y_observaciones (5...


## Paso 11 · Validar antes de cargar

**Errores** (si hay alguno, **no se carga nada**):
1. Campos obligatorios vacíos.
2. Notas fuera del rango 0 a 5 (lo mismo que exige el `CHECK` de la tabla).
3. Textos más largos de lo que permite la columna en SQL Server (si no, la carga fallaría a mitad de camino).

**Advertencias** (se informan, pero no detienen la carga, porque cada convenio puede tener reglas distintas):

4. Que los porcentajes de los componentes de cada corte sumen 100%.
5. Que la nota del corte coincida con el promedio ponderado de sus componentes.

In [41]:
LARGO_MAXIMO = {"curso_codigo": 30, "curso_nombre": 60, "asignatura": 40, "docente": 120,
                "programa": 100, "periodo": 60, "sede_jornada": 60, "tipo_documento": 5,
                "numero_documento": 20, "nombre_completo": 150, "subparametro": 60,
                "componente": 40, "promedio": 20, "archivo_origen": 255}

OBLIGATORIAS = ["curso_codigo", "curso_nombre", "asignatura", "programa", "periodo", "sede_jornada",
                "fecha_inicio", "fecha_fin", "tipo_documento", "numero_documento", "nombre_completo",
                "corte", "porcentaje_corte", "subparametro", "componente", "porcentaje_componente"]

errores, advertencias = [], []

# 1) obligatorias
for col in OBLIGATORIAS:
    vacios = final[col].isna().sum()
    if vacios:
        errores.append(f"'{col}' tiene {vacios} valores vacíos")

# 2) rango de notas
for col in ["nota", "nota_corte"]:
    fuera = final[(final[col] < 0) | (final[col] > 5)]
    if len(fuera):
        errores.append(f"'{col}' tiene {len(fuera)} valores fuera de 0-5")

# 3) largos
for col, maximo in LARGO_MAXIMO.items():
    largo = final[col].dropna().astype(str).str.len().max()
    if pd.notna(largo) and largo > maximo:
        errores.append(f"'{col}' tiene textos de {largo} caracteres (máximo {maximo})")

# 4) suma de porcentajes por estudiante y corte
grupo = ["curso_codigo", "tipo_documento", "numero_documento", "corte"]
suma = final.groupby(grupo)["porcentaje_componente"].sum()
if (suma.round(2) != 100).any():
    advertencias.append(f"{(suma.round(2) != 100).sum()} cortes cuyos componentes no suman 100%")

# 5) nota del corte vs. promedio ponderado de sus componentes
completos = final.groupby(grupo).filter(lambda g: g["nota"].notna().all() and g["nota_corte"].notna().all())
if len(completos):
    calc = completos.groupby(grupo).apply(
        lambda g: (g["nota"] * g["porcentaje_componente"]).sum() / g["porcentaje_componente"].sum(),
        include_groups=False)
    reportada = completos.groupby(grupo)["nota_corte"].first()
    distintas = ((calc - reportada).abs() > 0.05).sum()
    if distintas:
        advertencias.append(f"{distintas} cortes donde la nota reportada difiere del cálculo en más de 0.05")
    print(f"Revisé {len(reportada):,} cortes completos: la nota del corte coincide con el cálculo en "
          f"{len(reportada) - distintas:,}")

print("✅ Sin errores" if not errores else "❌ ERRORES:\n- " + "\n- ".join(errores))
print("⚠️ Advertencias:\n- " + "\n- ".join(advertencias) if advertencias else "Sin advertencias")

Revisé 6,363 cortes completos: la nota del corte coincide con el cálculo en 6,363
✅ Sin errores
Sin advertencias


## Paso 12 · Conectarse a SQL Server

Igual que en asistencia. Revisa que exista la tabla `dbo.notas_observaciones`.

In [42]:
engine = None

if CARGAR_EN_BD:
    import pyodbc
    from sqlalchemy import create_engine, text

    SERVIDOR = "bi-db-group.database.windows.net"
    BASE     = "rendimiento_academico"
    USUARIO  = "bi_admin"
    CONTRASENA_BD = "53/O~D&2m#o}"
    DRIVER   = "ODBC Driver 18 for SQL Server"

    clave_odbc = CONTRASENA_BD.replace("}", "}}")
    cadena = (
        f"DRIVER={{{DRIVER}}};SERVER={SERVIDOR},1433;DATABASE={BASE};"
        f"UID={USUARIO};PWD={{{clave_odbc}}};"
        "Encrypt=yes;TrustServerCertificate=yes;Connection Timeout=30;"
    )

    engine = create_engine("mssql+pyodbc://", creator=lambda: pyodbc.connect(cadena))

    with engine.connect() as conn:
        print("Conectado a:", conn.execute(text("SELECT @@SERVERNAME")).scalar(), "| base:", BASE)
        existe = conn.execute(text("SELECT OBJECT_ID('dbo.notas_observaciones', 'U')")).scalar()
        print("Tabla dbo.notas_observaciones:", "✅ existe" if existe else "❌ no existe")
else:
    print("CARGAR_EN_BD = False -> no me conecto (modo prueba)")

Conectado a: bi-db-group | base: rendimiento_academico
Tabla dbo.notas_observaciones: ✅ existe


## Paso 13 · Cargar con MERGE

Mismo método que en asistencia:

1. Subimos todo a una tabla temporal `#stg`.
2. `MERGE` de `#stg` contra `dbo.notas_observaciones`:

| Situación | Qué hace |
|---|---|
| La fila **no existe** | La **inserta** |
| Existe pero **cambió algo** (p. ej. la nota del Corte 2 pasó de vacía a 4.0, o cambió la observación) | La **actualiza** y pone `cargado_en = SYSDATETIME()` |
| Existe y es **igual** | No hace nada |

**Dos cosas nuevas respecto a asistencia:**

- **Comparar con `EXCEPT`.** Las notas pueden ser `NULL`, y en SQL `NULL <> 4.0` **no da verdadero** (da "desconocido"), así que un `<>` normal no detectaría que una nota pasó de vacía a 4.0. `EXISTS (SELECT s.… EXCEPT SELECT t.…)` sí trata los `NULL` como iguales entre sí y detecta el cambio.
- **`setinputsizes`.** Le decimos a pyodbc el tipo y tamaño exacto de cada columna. Sin esto, con observaciones de hasta 2.000 caracteres, la carga rápida (`fast_executemany`) puede fallar con *"String data, right truncation"*.

In [43]:
# Tipos de la tabla temporal (iguales a dbo.notas_observaciones)
TIPOS_SQL = {
    "curso_codigo": "VARCHAR(30)",        "curso_nombre": "NVARCHAR(60)",
    "asignatura": "NVARCHAR(40)",         "docente": "NVARCHAR(120)",
    "programa": "NVARCHAR(100)",          "periodo": "NVARCHAR(60)",
    "sede_jornada": "NVARCHAR(60)",       "fecha_inicio": "DATE",
    "fecha_fin": "DATE",                  "archivado": "BIT",
    "tipo_documento": "VARCHAR(5)",       "numero_documento": "VARCHAR(20)",
    "nombre_completo": "NVARCHAR(150)",   "corte": "TINYINT",
    "porcentaje_corte": "DECIMAL(5,2)",   "nota_corte": "DECIMAL(3,2)",
    "subparametro": "NVARCHAR(60)",       "componente": "NVARCHAR(40)",
    "porcentaje_componente": "DECIMAL(5,2)", "nota": "DECIMAL(3,2)",
    "promedio": "NVARCHAR(20)",           "acumulado_pct": "DECIMAL(5,2)",
    "inasistencias": "SMALLINT",          "observaciones": "NVARCHAR(MAX)",
    "archivo_origen": "NVARCHAR(255)",
}
COLUMNAS_CLAVE = ["curso_codigo", "tipo_documento", "numero_documento", "corte", "componente"]
NO_CLAVE = [c for c in COLUMNAS_TABLA if c not in COLUMNAS_CLAVE]

SQL_CREAR_STG = ("CREATE TABLE #stg (" +
                 ", ".join(f"{c} {TIPOS_SQL[c]} NULL" for c in COLUMNAS_TABLA) +
                 "); CREATE TABLE #acciones (accion NVARCHAR(10));")

SQL_INSERT_STG = (f"INSERT INTO #stg ({', '.join(COLUMNAS_TABLA)}) "
                  f"VALUES ({', '.join('?' for _ in COLUMNAS_TABLA)})" )

SQL_MERGE = f'''
MERGE dbo.notas_observaciones AS t
USING #stg AS s
   ON {" AND ".join(f"t.{c} = s.{c}" for c in COLUMNAS_CLAVE)}
WHEN MATCHED AND EXISTS (SELECT {", ".join("s." + c for c in NO_CLAVE)}
                         EXCEPT
                         SELECT {", ".join("t." + c for c in NO_CLAVE)}) THEN
    UPDATE SET {", ".join(f"t.{c} = s.{c}" for c in NO_CLAVE)},
               t.fecha_carga = SYSDATETIME()
WHEN NOT MATCHED BY TARGET THEN
    INSERT ({", ".join(COLUMNAS_TABLA)})
    VALUES ({", ".join("s." + c for c in COLUMNAS_TABLA)})
OUTPUT $action INTO #acciones;
'''
print(SQL_MERGE[:600], "...")


MERGE dbo.notas_observaciones AS t
USING #stg AS s
   ON t.curso_codigo = s.curso_codigo AND t.tipo_documento = s.tipo_documento AND t.numero_documento = s.numero_documento AND t.corte = s.corte AND t.componente = s.componente
WHEN MATCHED AND EXISTS (SELECT s.curso_nombre, s.asignatura, s.docente, s.programa, s.periodo, s.sede_jornada, s.fecha_inicio, s.fecha_fin, s.archivado, s.nombre_completo, s.porcentaje_corte, s.nota_corte, s.subparametro, s.porcentaje_componente, s.nota, s.promedio, s.acumulado_pct, s.inasistencias, s.observaciones, s.archivo_origen
                         EXCEPT
     ...


In [44]:
def tamano_pyodbc(tipo_sql):
    # 'NVARCHAR(60)' -> (pyodbc.SQL_WVARCHAR, 60, 0)  ;  'DECIMAL(3,2)' -> (pyodbc.SQL_DECIMAL, 3, 2)
    import pyodbc
    t = tipo_sql.upper()
    numeros = [int(n) for n in re.findall(r"\d+", t)]
    if t.startswith("NVARCHAR"):
        return (pyodbc.SQL_WVARCHAR, 0 if "MAX" in t else numeros[0], 0)   # 0 = MAX
    if t.startswith("VARCHAR"):
        return (pyodbc.SQL_VARCHAR, numeros[0], 0)
    if t.startswith("DECIMAL"):
        return (pyodbc.SQL_DECIMAL, numeros[0], numeros[1])
    return {"DATE": (pyodbc.SQL_TYPE_DATE, 0, 0), "BIT": (pyodbc.SQL_BIT, 0, 0),
            "TINYINT": (pyodbc.SQL_TINYINT, 0, 0), "SMALLINT": (pyodbc.SQL_SMALLINT, 0, 0)}[t]


def preparar_filas(tabla):
    # Convierte cada valor al tipo de Python que espera pyodbc (NaN -> None, float -> Decimal...)
    filas = []
    for registro in tabla[COLUMNAS_TABLA].itertuples(index=False):
        fila = []
        for col, valor in zip(COLUMNAS_TABLA, registro):
            if valor is None or valor is pd.NA or (isinstance(valor, float) and pd.isna(valor)):
                fila.append(None)
            elif TIPOS_SQL[col].startswith("DECIMAL"):
                fila.append(Decimal(str(round(float(valor), 2))))
            elif TIPOS_SQL[col] in ("TINYINT", "SMALLINT"):
                fila.append(int(valor))
            elif TIPOS_SQL[col] == "BIT":
                fila.append(bool(valor))
            else:
                fila.append(valor)
        filas.append(tuple(fila))
    return filas


def cargar(tabla, engine):
    filas = preparar_filas(tabla)
    conexion = engine.raw_connection()          # conexión pyodbc directa (una transacción)
    try:
        cursor = conexion.cursor()
        cursor.execute(SQL_CREAR_STG)
        cursor.fast_executemany = True
        cursor.setinputsizes([tamano_pyodbc(TIPOS_SQL[c]) for c in COLUMNAS_TABLA])
        cursor.executemany(SQL_INSERT_STG, filas)
        cursor.execute(SQL_MERGE)
        cursor.execute("SELECT accion, COUNT(*) FROM #acciones GROUP BY accion")
        resumen = {accion: n for accion, n in cursor.fetchall()}
        cursor.execute("DROP TABLE #stg; DROP TABLE #acciones;")
        conexion.commit()                       # todo o nada
    except Exception:
        conexion.rollback()
        raise
    finally:
        conexion.close()
    ins, act = resumen.get("INSERT", 0), resumen.get("UPDATE", 0)
    return ins, act, len(filas) - ins - act


if errores:
    print("❌ Hay errores de validación (paso 11): no se carga nada.")
elif engine is None:
    filas_prueba = preparar_filas(final.head(3))
    print("Modo prueba: así se envían las primeras filas a SQL Server:")
    for f in filas_prueba[:1]:
        print(dict(zip(COLUMNAS_TABLA, f)))
else:
    inicio = datetime.now()
    ins, act, igual = cargar(final, engine)
    segundos = (datetime.now() - inicio).total_seconds()
    print(f"✅ {ins:,} insertadas, {act:,} actualizadas, {igual:,} sin cambios ({segundos:.1f} s)")

✅ 64,880 insertadas, 0 actualizadas, 0 sin cambios (435.7 s)


## Paso 14 · Verificar en la base de datos

**14a.** Conteo por corte directamente en SQL Server.

**14b.** Primer cruce con asistencia: comparamos las **inasistencias que trae el reporte de notas** con las que **contamos en `dbo.asistencia`**. Fíjate en el patrón del cuaderno anterior: **primero resumimos** asistencia a una fila por estudiante y curso, **y después cruzamos**.

Que no coincidan todas es normal si los dos reportes se descargaron en días distintos, o si faltan cursos de asistencia por cargar.

In [45]:
if engine is not None:
    display(pd.read_sql('''
        SELECT corte,
               COUNT(*)                                    AS filas,
               COUNT(DISTINCT curso_codigo)                AS cursos,
               COUNT(DISTINCT numero_documento)            AS estudiantes,
               SUM(CASE WHEN nota IS NULL THEN 1 ELSE 0 END) AS sin_calificar,
               CAST(AVG(nota) AS DECIMAL(3,2))             AS promedio_notas
        FROM dbo.notas_observaciones
        GROUP BY corte
        ORDER BY corte
    ''', engine))
else:
    print("Modo prueba: cambia CARGAR_EN_BD = True en el paso 1 para cargar y verificar.")

,corte,filas,cursos,estudiantes,sin_calificar,promedio_notas
0,1,32440,217,6488,130,4.18
1,2,32440,217,6488,32287,4.56


In [46]:
if engine is not None:
    display(pd.read_sql('''
        WITH asis AS (          -- 1 fila por estudiante y curso
            SELECT curso_codigo, tipo_documento, numero_documento,
                   SUM(CASE WHEN estado LIKE 'I%' THEN 1 ELSE 0 END) AS inas_asistencia
            FROM dbo.asistencia
            GROUP BY curso_codigo, tipo_documento, numero_documento
        ),
        nota AS (               -- 1 fila por estudiante y curso
            SELECT DISTINCT curso_codigo, tipo_documento, numero_documento, inasistencias
            FROM dbo.notas_observaciones
        )
        SELECT COUNT(*)                                                  AS estudiantes_en_notas,
               SUM(CASE WHEN a.curso_codigo IS NULL THEN 1 ELSE 0 END)   AS sin_datos_de_asistencia,
               SUM(CASE WHEN a.inas_asistencia = n.inasistencias THEN 1 ELSE 0 END) AS inasistencias_coinciden,
               SUM(CASE WHEN a.inas_asistencia <> n.inasistencias THEN 1 ELSE 0 END) AS inasistencias_distintas
        FROM nota n
        LEFT JOIN asis a
               ON  a.curso_codigo     = n.curso_codigo
               AND a.tipo_documento   = n.tipo_documento
               AND a.numero_documento = n.numero_documento
    ''', engine))

,estudiantes_en_notas,sin_datos_de_asistencia,inasistencias_coinciden,inasistencias_distintas
0,6488,60,6427,1


---
### Resumen de lo que aprendiste en este cuaderno

- Un Excel que **ya viene en formato largo** no se gira: solo se **limpia y se convierte** columna por columna.
- **Un dato por columna:** `"Communication 1"` se separa en `componente` y `corte`.
- **Mismas claves en todas las tablas** (`curso_codigo`, `tipo_documento`, `numero_documento`) = cruces que sí funcionan.
- **Vacío no es cero:** las notas sin calificar quedan en `NULL`.
- **`EXCEPT` para comparar con `NULL`** en el `MERGE`.
- **`setinputsizes`** para que la carga rápida no falle con textos largos.